# From One Neuron to a Convolutional Net — Seen, Not Plotted

There are no loss curves in this notebook. Every idea is shown in the space where it actually lives: decision boundaries drawn on the data, loss as a landscape you walk across, hidden layers as a **warping of the plane**, and convolution as a window sliding over pixels with the feature map filling in behind it.

The thread is one repeated question — *what shape can this model carve?*

$$\text{one line}\ \longrightarrow\ \text{a line that moves}\ \longrightarrow\ \text{many lines folded together}\ \longrightarrow\ \text{lines that slide across an image}$$

Everything is plain NumPy, including the backpropagation, so nothing is hidden inside a library call. Models are trained once when a cell runs and the sliders scrub through the recorded history, which keeps every redraw instant.

In [2]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 108, "font.size": 9, "axes.titlesize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False,
})

RNG = np.random.default_rng(7)
CA, CB, CG = "#2166ac", "#b2182b", "#7f4fbf"      # class A, class B, accents
CMAP = mpl.colors.LinearSegmentedColormap.from_list("ab", [CA, "#f7f7f7", CB])


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


def relu(z):
    return np.maximum(z, 0.0)


def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def make_moons(n=300, noise=0.18, rng=RNG):
    k = n // 2
    t = np.linspace(0, np.pi, k)
    a = np.c_[np.cos(t), np.sin(t)]
    b = np.c_[1 - np.cos(t), 0.4 - np.sin(t)]
    X = np.vstack([a, b]) + rng.normal(0, noise, (2 * k, 2))
    X = (X - X.mean(0)) / X.std(0)
    return X, np.r_[np.zeros(k), np.ones(k)]


def make_circles(n=300, noise=0.13, rng=RNG):
    k = n // 2
    t = rng.uniform(0, 2 * np.pi, k)
    inner = np.c_[0.45 * np.cos(t), 0.45 * np.sin(t)]
    t2 = rng.uniform(0, 2 * np.pi, k)
    outer = np.c_[1.15 * np.cos(t2), 1.15 * np.sin(t2)]
    X = np.vstack([inner, outer]) + rng.normal(0, noise, (2 * k, 2))
    return X, np.r_[np.zeros(k), np.ones(k)]


def make_blobs(n=300, sep=1.7, rng=RNG):
    k = n // 2
    X = np.vstack([rng.normal([-sep / 2, -sep / 4], 0.62, (k, 2)),
                   rng.normal([sep / 2, sep / 4], 0.62, (k, 2))])
    return X - X.mean(0), np.r_[np.zeros(k), np.ones(k)]


def make_xor(n=300, rng=RNG):
    X = rng.uniform(-1.4, 1.4, (n, 2))
    return X, ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(float)


DATASETS = {"two blobs": make_blobs(), "two moons": make_moons(),
            "rings": make_circles(), "xor": make_xor()}


def bar_image(kind, size=14, noise=0.06, rng=RNG):
    """One image containing a bar at one of four orientations."""
    img = np.zeros((size, size))
    L = int(rng.integers(max(4, size // 2), size - 3))   # always fits
    r0 = int(rng.integers(1, size - L - 1))
    c0 = int(rng.integers(1, size - L - 1))
    if kind == 0:
        img[r0 + L // 2, c0:c0 + L] = 1.0
    elif kind == 1:
        img[r0:r0 + L, c0 + L // 2] = 1.0
    elif kind == 2:
        for i in range(L):
            img[r0 + i, c0 + i] = 1.0
    else:
        for i in range(L):
            img[r0 + i, c0 + L - 1 - i] = 1.0
    return img + rng.normal(0, noise, (size, size))


def bar_centered(kind, size=14, L=9, noise=0.06, rng=RNG):
    """The same four shapes, always in exactly the same place."""
    img = np.zeros((size, size))
    o, mid = (size - L) // 2, size // 2
    if kind == 0:
        img[mid, o:o + L] = 1.0
    elif kind == 1:
        img[o:o + L, mid] = 1.0
    elif kind == 2:
        for i in range(L):
            img[o + i, o + i] = 1.0
    else:
        for i in range(L):
            img[o + i, o + L - 1 - i] = 1.0
    return img + rng.normal(0, noise, (size, size))


def centered_dataset(n=600, noise=0.06, rng=RNG):
    y = rng.integers(0, 4, n)
    return np.stack([bar_centered(int(k), noise=noise, rng=rng) for k in y]), y


def bar_dataset(n=600, size=14, noise=0.06, rng=RNG):
    y = rng.integers(0, 4, n)
    X = np.stack([bar_image(int(k), size, noise, rng) for k in y])
    return X, y


SHAPE_NAMES = ["horizontal", "vertical", "diagonal /", "diagonal \\"]


def plane(ax, lim=2.6):
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])


def pixels(ax, img, cmap="gray", vmin=None, vmax=None, title=None):
    ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=8.5)


def montage(arrs, pad=1, pv=np.nan):
    """Tile equal-sized 2-D arrays into one image for a single imshow."""
    k = len(arrs)
    cols = int(np.ceil(np.sqrt(k)))
    rows = int(np.ceil(k / cols))
    h, w = arrs[0].shape
    out = np.full((rows * (h + pad) + pad, cols * (w + pad) + pad), pv)
    for i, a in enumerate(arrs):
        r, c = divmod(i, cols)
        out[pad + r * (h + pad):pad + r * (h + pad) + h,
            pad + c * (w + pad):pad + c * (w + pad) + w] = a
    return out


print(f"{len(DATASETS)} toy datasets ready | "
      f"bar images 14x14 in 4 orientations | pure NumPy, no ML library")

4 toy datasets ready | bar images 14x14 in 4 orientations | pure NumPy, no ML library


## 1 — A neuron is a line, and a squash

A single unit computes $\hat{y}=\sigma(\mathbf{w}\cdot\mathbf{x}+b)$, which does exactly two things. It **projects** every point onto the direction $\mathbf{w}$, collapsing the plane to one number, then it **squashes** that number into $(0,1)$.

So the whole model is one line. $\mathbf{w}$ is perpendicular to it and points toward class 1; $b$ slides it without rotating it; and $\|\mathbf{w}\|$ does not move the line at all — it only sharpens the transition, which you can see as the coloured band narrowing while the boundary stays put.

The left panel is the plane coloured by $\hat{y}$. The right panel is the same points after projection, sitting on the sigmoid. **A dataset is learnable by one neuron only if the two colours separate on that second panel** — try `two moons` and watch it become impossible no matter how you turn the line.

In [ ]:
def draw_neuron(dataset, angle, norm, b):
    X, y = DATASETS[dataset]
    w = norm * np.array([np.cos(np.deg2rad(angle)), np.sin(np.deg2rad(angle))])
    lim = 2.6
    g = np.linspace(-lim, lim, 260)
    GX, GY = np.meshgrid(g, g)
    P = sigmoid(w[0] * GX + w[1] * GY + b)

    fig = plt.figure(figsize=(11.0, 4.3))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.25], wspace=0.18,
                          left=0.04, right=0.97, top=0.86, bottom=0.12)

    a0 = fig.add_subplot(gs[0])
    a0.imshow(P, extent=[-lim, lim, -lim, lim], origin="lower", cmap=CMAP,
              vmin=0, vmax=1, alpha=0.9)
    a0.scatter(*X[y == 0].T, s=16, c=CA, edgecolor="w", linewidth=0.5, zorder=3)
    a0.scatter(*X[y == 1].T, s=16, c=CB, edgecolor="w", linewidth=0.5, zorder=3)
    d = np.array([-w[1], w[0]]) / max(np.linalg.norm(w), 1e-9)
    p0 = -b * w / max(w @ w, 1e-9)
    a0.plot(*np.c_[p0 - 6 * d, p0 + 6 * d], color="k", lw=2, zorder=4)
    a0.annotate("", xy=p0 + w / max(np.linalg.norm(w), 1e-9), xytext=p0,
                arrowprops=dict(arrowstyle="-|>", color="k", lw=2.2), zorder=5)
    a0.text(*(p0 + 1.18 * w / max(np.linalg.norm(w), 1e-9)), "$\\mathbf{w}$",
            fontsize=12, zorder=5)
    plane(a0, lim)
    a0.set_title("the plane, coloured by $\\hat{y}$")

    a1 = fig.add_subplot(gs[1])
    z = X @ w + b
    zs = np.linspace(min(-8, z.min() * 1.1), max(8, z.max() * 1.1), 400)
    a1.plot(zs, sigmoid(zs), color="0.35", lw=2, zorder=2)
    a1.axhline(0.5, color="0.75", lw=0.8, ls=":")
    a1.axvline(0, color="k", lw=1.5)
    for cls, col in ((0, CA), (1, CB)):
        m = y == cls
        a1.scatter(z[m], sigmoid(z[m]), s=16, c=col, edgecolor="w",
                   linewidth=0.5, zorder=3)
        a1.scatter(z[m], np.full(m.sum(), -0.09) + 0.03 * cls, s=8, c=col,
                   alpha=0.5, zorder=3)
    acc = ((sigmoid(z) > 0.5) == (y == 1)).mean()
    a1.set_ylim(-0.16, 1.05); a1.set_yticks([0, 0.5, 1])
    a1.set_xlabel("projection  $\\mathbf{w}\\cdot\\mathbf{x}+b$")
    a1.set_ylabel("$\\hat{y}$")
    a1.set_title(f"project, then squash — accuracy {acc:.1%}"
                 + ("   (separated)" if acc > 0.97 else "   (overlapping)"))
    plt.show()


w1 = dict(dataset=widgets.Dropdown(options=list(DATASETS), value="two blobs",
                                   description="data:",
                                   style={"description_width": "88px"},
                                   layout=widgets.Layout(width="300px")),
          angle=widgets.FloatSlider(value=30, min=0, max=360, step=5,
                                    description="direction of w:",
                                    style={"description_width": "112px"},
                                    layout=widgets.Layout(width="330px"),
                                    continuous_update=False),
          norm=widgets.FloatSlider(value=2.0, min=0.2, max=12.0, step=0.2,
                                   description="‖w‖ sharpness:",
                                   style={"description_width": "112px"},
                                   layout=widgets.Layout(width="330px"),
                                   continuous_update=False),
          b=widgets.FloatSlider(value=0.0, min=-4, max=4, step=0.1,
                                description="bias b:",
                                style={"description_width": "112px"},
                                layout=widgets.Layout(width="330px"),
                                continuous_update=False))
display(widgets.VBox([widgets.HBox([w1["dataset"], w1["angle"]]),
                      widgets.HBox([w1["norm"], w1["b"]])]),
        widgets.interactive_output(draw_neuron, w1))

Output()

## 2 — Learning is a walk on a landscape

Turning the line by hand does not scale. Instead, score every possible line with a loss and roll downhill:

$$\mathcal{L}_{\text{CE}}=-\frac{1}{N}\sum_i\big[y_i\log\hat{y}_i+(1-y_i)\log(1-\hat{y}_i)\big],
\qquad \mathbf{w}\leftarrow\mathbf{w}-\eta\nabla\mathcal{L}$$

The data is centred and the bias dropped, so there are exactly two parameters and the left panel is the **true, complete loss surface** — not a slice through a higher-dimensional one. Every point on it is a line you could have drawn; the walk is the model learning.

How big a step you can survive is decided by the *shape* of that surface, and the two losses here differ completely:

| loss | surface | largest survivable step |
|---|---|---|
| cross-entropy | flattens far from the data, gradient bounded by $\lvert\sigma(z)-y\rvert\le1$ | **none** — even $\eta=30$ takes one wild leap and then settles |
| squared error | a quadratic bowl with curvatures $\lambda=0.32$ and $1.27$ | hard limit at $\eta=2/\lambda_{\max}=1.58$ |

Take squared error and step through $\eta=1.5$: the path zigzags across the narrow axis of the bowl and still arrives. Nudge to $\eta=1.9$ and every step lands further out than the last. Now switch to cross-entropy at $\eta=30$ and watch it survive a step that annihilates the quadratic. Notice too that the walk slows near the bottom without being told to — the gradient shrinks as the surface flattens.

In [3]:
_Xb, _yb = DATASETS["two blobs"]
_tb = 2 * _yb - 1
LOSSES = ["cross-entropy", "squared error"]
LRS = [0.05, 0.5, 1.5, 1.9, 30.0]
_gw = np.linspace(-4.5, 4.5, 121)
_GW1, _GW2 = np.meshgrid(_gw, _gw)


def grad(w, loss):
    if loss == "cross-entropy":
        return _Xb.T @ (sigmoid(_Xb @ w) - _yb) / len(_yb)
    return _Xb.T @ (_Xb @ w - _tb) / len(_yb)


def loss_at(w, loss):
    z = _Xb @ w
    if loss == "cross-entropy":
        p = sigmoid(z)
        return float(-(_yb * np.log(p + 1e-12)
                       + (1 - _yb) * np.log(1 - p + 1e-12)).mean())
    return float(0.5 * ((z - _tb) ** 2).mean())


def gd_path(loss, lr, steps=140, w0=(2.6, -2.4)):
    w = np.array(w0, dtype=float)
    P = [w.copy()]
    for _ in range(steps):
        w = w - lr * grad(w, loss)
        w = np.clip(np.nan_to_num(w, nan=60.0, posinf=60.0, neginf=-60.0), -60, 60)
        P.append(w.copy())
    return np.array(P)


def _surface(loss):
    Z = _Xb @ np.stack([_GW1.ravel(), _GW2.ravel()])
    if loss == "cross-entropy":
        p = sigmoid(Z)
        L = -(_yb[:, None] * np.log(p + 1e-12)
              + (1 - _yb)[:, None] * np.log(1 - p + 1e-12)).mean(0)
    else:
        L = 0.5 * ((Z - _tb[:, None]) ** 2).mean(0)
    return L.reshape(_GW1.shape)


SURF = {ln: _surface(ln) for ln in LOSSES}
PATHS = {(ln, lr): gd_path(ln, lr) for ln in LOSSES for lr in LRS}
CURV = np.linalg.eigvalsh(_Xb.T @ _Xb / len(_yb))


def draw_descent(loss, lr, step):
    P = PATHS[(loss, lr)]
    step = min(step, len(P) - 1)
    w = P[step]
    S = SURF[loss]
    lim = 2.6

    fig = plt.figure(figsize=(11.2, 4.3))
    gs = fig.add_gridspec(1, 2, wspace=0.16, left=0.04, right=0.97,
                          top=0.84, bottom=0.12)

    a0 = fig.add_subplot(gs[0])
    lv = np.linspace(S.min(), np.percentile(S, 92), 26)
    a0.contourf(_GW1, _GW2, np.clip(S, None, lv[-1]), levels=lv, cmap="magma")
    a0.contour(_GW1, _GW2, np.clip(S, None, lv[-1]), levels=lv[::3], colors="w",
               linewidths=0.4, alpha=0.5)
    a0.plot(P[:step + 1, 0], P[:step + 1, 1], color="w", lw=1.6, zorder=3)
    a0.scatter(P[:step + 1:max(step // 24, 1), 0],
               P[:step + 1:max(step // 24, 1), 1], s=14, c="w", zorder=4)
    a0.scatter([w[0]], [w[1]], s=110, c=CG, edgecolor="w", linewidth=1.6, zorder=5)
    a0.set_xlim(-4.5, 4.5); a0.set_ylim(-4.5, 4.5); a0.set_aspect("equal")
    a0.set_xticks([]); a0.set_yticks([])
    a0.set_xlabel("$w_1$"); a0.set_ylabel("$w_2$")
    blew = np.linalg.norm(P[-1]) > 20
    a0.set_title(f"{loss} surface — step {step}/{len(P) - 1}"
                 + ("   DIVERGING" if blew else ""))

    a1 = fig.add_subplot(gs[1])
    g = np.linspace(-lim, lim, 240)
    GX, GY = np.meshgrid(g, g)
    zz = w[0] * GX + w[1] * GY
    field = sigmoid(zz) if loss == "cross-entropy" else np.clip((zz + 1) / 2, 0, 1)
    a1.imshow(field, extent=[-lim, lim, -lim, lim], origin="lower", cmap=CMAP,
              vmin=0, vmax=1, alpha=0.9)
    a1.scatter(*_Xb[_yb == 0].T, s=16, c=CA, edgecolor="w", linewidth=0.5, zorder=3)
    a1.scatter(*_Xb[_yb == 1].T, s=16, c=CB, edgecolor="w", linewidth=0.5, zorder=3)
    nrm = max(np.linalg.norm(w), 1e-9)
    d = np.array([-w[1], w[0]]) / nrm
    a1.plot(*np.c_[-6 * d, 6 * d], color="k", lw=2, zorder=4)
    plane(a1, lim)
    acc = (((_Xb @ w) > 0) == (_yb == 1)).mean()
    a1.set_title(f"the line this point encodes — loss {loss_at(w, loss):.3f}, "
                 f"accuracy {acc:.1%}")
    plt.show()


w2 = dict(loss=widgets.Dropdown(options=LOSSES, value="squared error",
                                description="loss:",
                                style={"description_width": "72px"},
                                layout=widgets.Layout(width="260px")),
          lr=widgets.Dropdown(options=LRS, value=0.5, description="step size η:",
                              style={"description_width": "96px"},
                              layout=widgets.Layout(width="260px")),
          step=widgets.IntSlider(value=0, min=0, max=140, step=1,
                                 description="gradient step:",
                                 style={"description_width": "112px"},
                                 layout=widgets.Layout(width="400px"),
                                 continuous_update=False))
display(widgets.HBox([w2["loss"], w2["lr"], w2["step"]]),
        widgets.interactive_output(draw_descent, w2))

Output()

## 3 — A hidden layer bends the plane

One line cannot cut the moons apart, and no amount of descending fixes that — the shape is wrong, not the position. A hidden layer changes the shape by drawing $H$ lines at once and then bending the space along them:

$$\mathbf{h}=\tanh(W_1\mathbf{x}+\mathbf{b}_1),\qquad \hat{y}=\sigma(\mathbf{w}_2\cdot\mathbf{h}+b_2)$$

The middle panel is the point of the section. It shows the data **after** the hidden layer has moved it, and the output neuron is still just one line — but now it is drawn in that new, warped space. The network does not learn a curved boundary; it learns a *straight* boundary in coordinates it invented so that straight is enough.

Each hidden unit contributes one fold, drawn as a dashed line on the left. The moons are forgiving — one fold already scores 89%, because a single line does catch most of them and the failure shows in the *shape* of the region rather than in the number. Switch to `rings` or `xor` to see one fold genuinely helpless at 67% and 72%; three folds take both to 96–100%. Past that, extra folds buy almost nothing and the boundary starts detouring around individual noise points.

In [3]:
def train_mlp(X, y, H, steps=2500, lr=0.6, seed=0):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 1.0, (2, H)); b1 = np.zeros(H)
    W2 = rng.normal(0, 1.0, H); b2 = 0.0
    for _ in range(steps):
        Hh = np.tanh(X @ W1 + b1)
        p = sigmoid(Hh @ W2 + b2)
        d = (p - y) / len(y)
        gW2, gb2 = Hh.T @ d, d.sum()
        dh = np.outer(d, W2) * (1 - Hh ** 2)
        W1 -= lr * (X.T @ dh); b1 -= lr * dh.sum(0)
        W2 -= lr * gW2; b2 -= lr * gb2
    return W1, b1, W2, b2


HS = [1, 2, 3, 4, 6, 12]
MLPS = {(dn, h): train_mlp(*DATASETS[dn], h)
        for dn in ("two moons", "rings", "xor") for h in HS}


def draw_hidden(dataset, H, show_folds):
    X, y = DATASETS[dataset]
    W1, b1, W2, b2 = MLPS[(dataset, H)]
    lim = 2.6 if dataset == "two moons" else 2.0

    fig = plt.figure(figsize=(12.4, 4.2))
    gs = fig.add_gridspec(1, 3, wspace=0.16, left=0.03, right=0.98,
                          top=0.84, bottom=0.08)

    a0 = fig.add_subplot(gs[0])
    a0.scatter(*X[y == 0].T, s=16, c=CA, edgecolor="w", linewidth=0.5, zorder=3)
    a0.scatter(*X[y == 1].T, s=16, c=CB, edgecolor="w", linewidth=0.5, zorder=3)
    if show_folds:
        for j in range(H):
            wj, bj = W1[:, j], b1[j]
            nj = max(np.linalg.norm(wj), 1e-9)
            dj = np.array([-wj[1], wj[0]]) / nj
            pj = -bj * wj / max(wj @ wj, 1e-9)
            a0.plot(*np.c_[pj - 9 * dj, pj + 9 * dj], color=CG, lw=1.3,
                    ls="--", alpha=0.85, zorder=2)
    plane(a0, lim)
    a0.set_title(f"input space — {H} hidden fold(s)")

    a1 = fig.add_subplot(gs[1])
    Hh = np.tanh(X @ W1 + b1)
    if H == 1:
        co = np.c_[Hh[:, 0], np.zeros(len(Hh))]
        lab = "hidden unit 1 (only one axis exists)"
    else:
        Hc = Hh - Hh.mean(0)
        U, S, Vt = np.linalg.svd(Hc, full_matrices=False)
        co = Hc @ Vt[:2].T
        lab = ("hidden space" if H == 2 else
               "hidden space (top 2 principal directions)")
    a1.scatter(*co[y == 0].T, s=16, c=CA, edgecolor="w", linewidth=0.5)
    a1.scatter(*co[y == 1].T, s=16, c=CB, edgecolor="w", linewidth=0.5)
    a1.axhline(0, color="0.85", lw=0.8); a1.axvline(0, color="0.85", lw=0.8)
    a1.set_xticks([]); a1.set_yticks([]); a1.set_aspect("equal")
    a1.set_title(lab)

    a2 = fig.add_subplot(gs[2])
    g = np.linspace(-lim, lim, 220)
    GX, GY = np.meshgrid(g, g)
    P = sigmoid(np.tanh(np.c_[GX.ravel(), GY.ravel()] @ W1 + b1) @ W2 + b2)
    a2.imshow(P.reshape(GX.shape), extent=[-lim, lim, -lim, lim], origin="lower",
              cmap=CMAP, vmin=0, vmax=1, alpha=0.92)
    a2.contour(GX, GY, P.reshape(GX.shape), levels=[0.5], colors="k",
               linewidths=2)
    a2.scatter(*X[y == 0].T, s=14, c=CA, edgecolor="w", linewidth=0.4, zorder=3)
    a2.scatter(*X[y == 1].T, s=14, c=CB, edgecolor="w", linewidth=0.4, zorder=3)
    plane(a2, lim)
    acc = ((sigmoid(np.tanh(X @ W1 + b1) @ W2 + b2) > 0.5) == (y == 1)).mean()
    a2.set_title(f"decision region — accuracy {acc:.1%}")
    plt.show()


w3 = dict(dataset=widgets.Dropdown(options=["two moons", "rings", "xor"],
                                   value="two moons", description="data:",
                                   style={"description_width": "88px"},
                                   layout=widgets.Layout(width="290px")),
          H=widgets.SelectionSlider(options=HS, value=3,
                                    description="hidden units:",
                                    style={"description_width": "104px"},
                                    layout=widgets.Layout(width="360px"),
                                    continuous_update=False),
          show_folds=widgets.Checkbox(value=True, description="show folds",
                                      indent=False))
display(widgets.HBox([w3["dataset"], w3["H"], w3["show_folds"]]),
        widgets.interactive_output(draw_hidden, w3))

Output()

## 4 — Why this stops working on images

Feed a $14\times14$ image to a dense layer and the first thing that happens is `reshape(196)`. The rows are laid end to end, and two pixels that were vertical neighbours end up 14 apart in the vector — the layer has no way of knowing they were ever adjacent. Any permutation of the pixels would train equally well.

The consequence is the second row below. A dense classifier trained on bars in one place learns *those pixels*, so its weights are a photograph of the training set. Shift the same bar two pixels and the evidence lands on weights that were never trained, and the prediction falls apart even though a human sees no change at all.

$$\text{dense: } 196\times H \text{ weights, position-specific}
\qquad\text{vs.}\qquad
\text{conv: } 5\times 5=25 \text{ weights, reused everywhere}$$

Drag the shift and watch the bars on the right collapse. This failure is the entire argument for convolution.

In [4]:
_Xs, _ys = centered_dataset(700)
_Xtr, _ytr = _Xs[:500].reshape(500, -1), _ys[:500]
_W = np.zeros((196, 4)); _bb = np.zeros(4)
_Y1 = np.eye(4)[_ytr]
for _ in range(600):
    _P = softmax(_Xtr @ _W + _bb)
    _d = (_P - _Y1) / len(_ytr)
    _W -= 0.5 * (_Xtr.T @ _d); _bb -= 0.5 * _d.sum(0)
DENSE_ACC = (np.argmax(softmax(_Xs[500:].reshape(-1, 196) @ _W + _bb), 1)
             == _ys[500:]).mean()


def shift_img(img, dx, dy):
    """Translate with zero fill — no wrap-around artefacts."""
    out = np.zeros_like(img)
    H, W = img.shape
    sy0, sy1 = max(0, -dy), min(H, H - dy)
    sx0, sx1 = max(0, -dx), min(W, W - dx)
    out[sy0 + dy:sy1 + dy, sx0 + dx:sx1 + dx] = img[sy0:sy1, sx0:sx1]
    return out


def draw_flatten(idx, dx, dy):
    img = _Xs[500 + idx]
    sh = shift_img(img, dx, dy)
    p0 = softmax(img.reshape(1, -1) @ _W + _bb)[0]
    p1 = softmax(sh.reshape(1, -1) @ _W + _bb)[0]
    true = _ys[500 + idx]

    fig = plt.figure(figsize=(12.4, 5.0))
    gs = fig.add_gridspec(2, 4, height_ratios=[1, 0.85], hspace=0.42,
                          wspace=0.28, left=0.04, right=0.97, top=0.88,
                          bottom=0.06)

    pixels(fig.add_subplot(gs[0, 0]), img, title=f"original — {SHAPE_NAMES[true]}")
    pixels(fig.add_subplot(gs[0, 1]), sh, title=f"shifted by ({dx:+d}, {dy:+d})")

    a2 = fig.add_subplot(gs[0, 2])
    a2.imshow(np.vstack([img.ravel(), sh.ravel()]), aspect="auto", cmap="gray",
              interpolation="nearest")
    a2.set_yticks([0, 1]); a2.set_yticklabels(["original", "shifted"], fontsize=8)
    a2.set_xticks([]); a2.set_title("what the layer receives:\n196 numbers in a row",
                                    fontsize=9)

    for c in range(4):
        pixels(fig.add_subplot(gs[1, c]) if c < 3 else fig.add_subplot(gs[1, 3]),
               _W[:, c].reshape(14, 14), cmap="RdBu_r",
               vmin=-abs(_W).max(), vmax=abs(_W).max(),
               title=f"weights for '{SHAPE_NAMES[c]}'")

    a3 = fig.add_subplot(gs[0, 3])
    xx = np.arange(4)
    a3.barh(xx - 0.2, p0, height=0.36, color="0.6", label="original")
    a3.barh(xx + 0.2, p1, height=0.36, color=CB, label="shifted")
    a3.set_yticks(xx); a3.set_yticklabels(SHAPE_NAMES, fontsize=8)
    a3.set_xlim(0, 1); a3.invert_yaxis(); a3.legend(fontsize=7)
    a3.set_title(f"confidence  (unshifted test acc {DENSE_ACC:.0%}, "
                 f"chance 25%)", fontsize=9)
    plt.show()


w4 = dict(idx=widgets.IntSlider(value=3, min=0, max=100, step=1,
                                description="test image:",
                                style={"description_width": "96px"},
                                layout=widgets.Layout(width="330px"),
                                continuous_update=False),
          dx=widgets.IntSlider(value=2, min=-4, max=4, step=1,
                               description="shift x:",
                               style={"description_width": "96px"},
                               layout=widgets.Layout(width="330px"),
                               continuous_update=False),
          dy=widgets.IntSlider(value=0, min=-4, max=4, step=1,
                               description="shift y:",
                               style={"description_width": "96px"},
                               layout=widgets.Layout(width="330px"),
                               continuous_update=False))
display(widgets.HBox([w4["idx"], w4["dx"], w4["dy"]]),
        widgets.interactive_output(draw_flatten, w4))

Output()

## 5 — Convolution: one small window, reused everywhere

A convolution keeps the grid. A $k\times k$ patch of weights is laid on the image, multiplied, summed into a single output pixel, then **slid** and applied again — the same 25 numbers at every position:

$$O[i,j]=\sum_{u,v}K[u,v]\,I[i s+u,\ j s+v],\qquad
O_{\text{size}}=\left\lfloor\frac{N+2p-k}{s}\right\rfloor+1$$

Because the weights are reused, a feature learned in one corner is detected in every corner for free — the position-dependence of section 4 is gone by construction.

Scrub `window position` to watch the output fill in behind the window one pixel at a time. Then change the kernel and read what each one is measuring: `edge ↕` responds where brightness changes vertically, `blur` averages its neighbourhood, `sharpen` amplifies the difference from it. The right-hand panel shows why depth matters — stacking $3\times3$ convolutions grows the **receptive field** as $1+2L$, so a pixel deep in the network sees a wide patch of the original image while still costing only nine weights per layer.

In [5]:
KERNELS = {
    "edge ↕ (Sobel y)": np.array([[-1., -2, -1], [0, 0, 0], [1, 2, 1]]),
    "edge ↔ (Sobel x)": np.array([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "blur (box)": np.ones((3, 3)) / 9,
    "sharpen": np.array([[0., -1, 0], [-1, 5, -1], [0, -1, 0]]),
    "identity": np.array([[0., 0, 0], [0, 1, 0], [0, 0, 0]]),
    "diagonal": np.array([[2., 1, 0], [1, 0, -1], [0, -1, -2]]),
}
_CIMG = np.clip(sum(bar_image(k, 18) for k in range(4)), 0, 1.4)


def conv2d(img, K, stride=1, pad=0):
    if pad:
        img = np.pad(img, pad)
    k = K.shape[0]
    n = (img.shape[0] - k) // stride + 1
    out = np.zeros((n, n))
    for u in range(k):
        for v in range(k):
            out += K[u, v] * img[u:u + n * stride:stride, v:v + n * stride:stride]
    return out, img


def draw_conv(kernel, stride, pad, pos_pct):
    K = KERNELS[kernel]
    out, padded = conv2d(_CIMG, K, stride, pad)
    n = out.shape[0]
    idx = int(round(pos_pct / 100 * (n * n - 1)))
    r, c = divmod(idx, n)

    fig = plt.figure(figsize=(12.6, 4.0))
    gs = fig.add_gridspec(1, 4, width_ratios=[1.15, 0.62, 1.15, 1.0],
                          wspace=0.26, left=0.03, right=0.98, top=0.84,
                          bottom=0.06)

    a0 = fig.add_subplot(gs[0])
    pixels(a0, padded, title=f"input {padded.shape[0]}×{padded.shape[0]}"
                             + (f"  (padded by {pad})" if pad else ""))
    a0.add_patch(plt.Rectangle((c * stride - 0.5, r * stride - 0.5), 3, 3,
                               fill=False, ec=CB, lw=2.5))

    a1 = fig.add_subplot(gs[1])
    pixels(a1, K, cmap="RdBu_r", vmin=-abs(K).max(), vmax=abs(K).max(),
           title="kernel")
    for (i, j), v in np.ndenumerate(K):
        a1.text(j, i, f"{v:g}", ha="center", va="center", fontsize=8,
                color="k" if abs(v) < abs(K).max() * 0.6 else "w")

    a2 = fig.add_subplot(gs[2])
    partial = np.full_like(out, np.nan)
    partial.ravel()[:idx + 1] = out.ravel()[:idx + 1]
    pixels(a2, partial, cmap="viridis", vmin=out.min(), vmax=out.max(),
           title=f"feature map {n}×{n} — filling in ({idx + 1}/{n * n})")
    a2.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, ec=CB,
                               lw=2.5))
    patch = padded[r * stride:r * stride + 3, c * stride:c * stride + 3]
    a2.set_xlabel(f"this pixel = Σ(patch × kernel) = {(patch * K).sum():+.2f}",
                  fontsize=8)

    a3 = fig.add_subplot(gs[3])
    a3.imshow(np.zeros((15, 15)), cmap="gray", vmin=0, vmax=1)
    for L, col in zip((1, 2, 3), (CA, CG, CB)):
        s = 1 + 2 * L
        a3.add_patch(plt.Rectangle((7 - s / 2, 7 - s / 2), s, s, fill=False,
                                   ec=col, lw=2.2))
        a3.text(7 + s / 2 + 0.2, 7 - s / 2, f"{L} layer{'s' if L > 1 else ''}: "
                                            f"{s}×{s}", color=col, fontsize=8,
                va="center")
    a3.plot([7], [7], "s", color="w", ms=5)
    a3.set_xticks([]); a3.set_yticks([]); a3.set_xlim(-0.5, 20)
    a3.set_title("receptive field of one deep pixel\nstacked 3×3 convs: 1 + 2L")
    plt.show()


w5 = dict(kernel=widgets.Dropdown(options=list(KERNELS), value="edge ↕ (Sobel y)",
                                  description="kernel:",
                                  style={"description_width": "80px"},
                                  layout=widgets.Layout(width="300px")),
          stride=widgets.IntSlider(value=1, min=1, max=3, step=1,
                                   description="stride:",
                                   style={"description_width": "80px"},
                                   layout=widgets.Layout(width="260px"),
                                   continuous_update=False),
          pad=widgets.IntSlider(value=0, min=0, max=2, step=1,
                                description="padding:",
                                style={"description_width": "80px"},
                                layout=widgets.Layout(width="260px"),
                                continuous_update=False),
          pos_pct=widgets.FloatSlider(value=45, min=0, max=100, step=1,
                                      description="window position:",
                                      style={"description_width": "112px"},
                                      layout=widgets.Layout(width="420px"),
                                      continuous_update=False))
display(widgets.VBox([widgets.HBox([w5["kernel"], w5["stride"], w5["pad"]]),
                      widgets.HBox([w5["pos_pct"]])]),
        widgets.interactive_output(draw_conv, w5))

Output()

## 6 — What a CNN learns when nobody tells it what to look for

Section 5 used kernels chosen by hand. Here four $5\times5$ kernels start as random noise and are learned by backpropagation, on bars at four orientations scattered anywhere in the frame:

$$\text{input }14\times14\ \to\ \text{conv }4@5\times5\ \to\ \text{ReLU}\ \to\ \text{maxpool }2\times2\ \to\ \text{dense}\ \to\ 4\text{ classes}$$

Scrub `training epoch` from 0 and watch the top-left panel resolve out of static. Nobody specified edges; oriented filters are simply what the gradient found. Measured on the trained network: the average kernel's correlation with its best-matching bar template climbs from $0.34$ at initialisation to $0.65$, the kernels move $120\%$ from where they started, and **none of the four is decorative** — zeroing any single one costs between 20 and 58 points of accuracy.

One detail worth not glossing over: there is no dedicated "horizontal" kernel. The four filters settle on vertical and the two diagonals, and horizontal bars are recognised partly by which filters stay *silent*. Features are read as a population, not one-per-class — which is exactly why reading meaning into individual units is so unreliable in larger networks.

The lower panels follow one image through. Each feature map lights up only where its kernel matches the local pattern, ReLU discards the negative half, and pooling throws away precise position while keeping the evidence.

In [ ]:
def cnn_forward(X, Kc, bc, Wd, bd):
    N, S, _ = X.shape
    k = Kc.shape[0]
    n = S - k + 1
    Z = np.zeros((N, n, n, Kc.shape[2]))
    for u in range(k):
        for v in range(k):
            Z += X[:, u:u + n, v:v + n, None] * Kc[u, v][None, None, None, :]
    Z = Z + bc
    A = relu(Z)
    m = n // 2
    B = A[:, :2 * m, :2 * m, :].reshape(N, m, 2, m, 2, -1)
    P = B.max(axis=(2, 4))
    F = P.reshape(N, -1)
    return Z, A, P, softmax(F @ Wd + bd)


def train_cnn(X, y, K=4, k=5, epochs=30, lr=0.08, seed=3):
    rng = np.random.default_rng(seed)
    S = X.shape[1]; n = S - k + 1; m = n // 2
    Kc = rng.normal(0, 0.25, (k, k, K)); bc = np.zeros(K)
    Wd = rng.normal(0, 0.12, (m * m * K, 4)); bd = np.zeros(4)
    Y = np.eye(4)[y]
    snaps = [(Kc.copy(), bc.copy(), Wd.copy(), bd.copy(), np.nan)]
    for ep in range(epochs):
        order = rng.permutation(len(X))
        for s in range(0, len(X), 32):
            b = order[s:s + 32]
            xb, yb = X[b], Y[b]
            Z, A, P, out = cnn_forward(xb, Kc, bc, Wd, bd)
            d = (out - yb) / len(b)
            gWd = P.reshape(len(b), -1).T @ d
            gbd = d.sum(0)
            dP = (d @ Wd.T).reshape(P.shape)
            dA = np.zeros_like(A)
            for i in range(m):
                for j in range(m):
                    win = A[:, 2 * i:2 * i + 2, 2 * j:2 * j + 2, :]
                    mask = win == win.max(axis=(1, 2), keepdims=True)
                    mask = mask / np.maximum(mask.sum(axis=(1, 2), keepdims=True), 1)
                    dA[:, 2 * i:2 * i + 2, 2 * j:2 * j + 2, :] += \
                        mask * dP[:, i, j, :][:, None, None, :]
            dZ = dA * (Z > 0)
            gK = np.zeros_like(Kc)
            for u in range(k):
                for v in range(k):
                    gK[u, v] = np.einsum("nij,nijc->c",
                                         xb[:, u:u + n, v:v + n], dZ)
            Kc -= lr * gK
            bc -= lr * dZ.sum(axis=(0, 1, 2))
            Wd -= lr * gWd; bd -= lr * gbd
        acc = (np.argmax(cnn_forward(X, Kc, bc, Wd, bd)[3], 1) == y).mean()
        snaps.append((Kc.copy(), bc.copy(), Wd.copy(), bd.copy(), acc))
    return snaps


_Xc, _yc = bar_dataset(700, noise=0.30)
CNN = train_cnn(_Xc, _yc)


def draw_cnn(epoch, sample):
    Kc, bc, Wd, bd, acc = CNN[epoch]
    x = _Xc[sample:sample + 1]
    Z, A, P, out = cnn_forward(x, Kc, bc, Wd, bd)
    K = Kc.shape[2]
    kern = [Kc[:, :, i] for i in range(K)]
    maps = [A[0, :, :, i] for i in range(K)]
    pool = [P[0, :, :, i] for i in range(K)]

    fig = plt.figure(figsize=(12.4, 5.0))
    gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1],
                          hspace=0.32, wspace=0.22, left=0.03, right=0.98,
                          top=0.88, bottom=0.05)

    a0 = fig.add_subplot(gs[0, 0])
    v = max(abs(Kc).max(), 1e-9)
    pixels(a0, montage(kern), cmap="RdBu_r", vmin=-v, vmax=v,
           title=f"the 4 learned kernels — epoch {epoch}")

    a1 = fig.add_subplot(gs[1, 0])
    pixels(a1, x[0], title=f"input: {SHAPE_NAMES[_yc[sample]]}")

    a2 = fig.add_subplot(gs[0, 1:3])
    pixels(a2, montage(maps), cmap="viridis",
           title="feature maps after conv + ReLU  (10×10 each)")

    a3 = fig.add_subplot(gs[1, 1:3])
    pixels(a3, montage(pool), cmap="viridis",
           title="after 2×2 max-pool  (5×5 each) — position blurred, "
                 "evidence kept")

    a4 = fig.add_subplot(gs[:, 3])
    xx = np.arange(4)
    a4.barh(xx, out[0], color=[CB if i == np.argmax(out[0]) else "0.7"
                               for i in range(4)])
    a4.set_yticks(xx); a4.set_yticklabels(SHAPE_NAMES, fontsize=9)
    a4.set_xlim(0, 1); a4.invert_yaxis()
    a4.set_title(f"prediction\ntrain accuracy "
                 + ("—" if not np.isfinite(acc) else f"{acc:.1%}"), fontsize=9)
    plt.show()


w6 = dict(epoch=widgets.IntSlider(value=0, min=0, max=len(CNN) - 1, step=1,
                                  description="training epoch:",
                                  style={"description_width": "112px"},
                                  layout=widgets.Layout(width="430px"),
                                  continuous_update=False),
          sample=widgets.IntSlider(value=0, min=0, max=200, step=1,
                                   description="image:",
                                   style={"description_width": "96px"},
                                   layout=widgets.Layout(width="330px"),
                                   continuous_update=False))
display(widgets.HBox([w6["epoch"], w6["sample"]]),
        widgets.interactive_output(draw_cnn, w6))

Output()